# Week 30, Day 2: Fitting the gates: why n to the fourth power

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How much charge moves?

Compute n∞(V) from the HH rate functions for V from -100 to +40 mV, fit a Boltzmann curve by grid search over V_half from -80 to 0 mV and k from 5 to 30 mV, and convert the slope factor into an effective charge `z` per particle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rates_n(V):
    V = V + 1e-7
    return 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)), 0.125 * np.exp(-(V + 65) / 80)

V = np.arange(-100, 41, 1.0)
an, bn = rates_n(V)
ninf = an / (an + bn)
best = (np.inf, None, None)
for vh in np.arange(-80, 0.1, 0.5):
    for k in np.arange(5, 30.01, 0.1):
        err = np.sum((ninf - 1 / (1 + np.exp(-(V - vh) / k))) ** 2)
        if err < best[0]:
            best = (err, vh, k)
_, V_half, k_slope = best
z = ...
print(f"n_inf: V_half {V_half:.1f} mV, slope factor {k_slope:.1f} mV, about {z:.2f} elementary charges per particle")
plt.plot(V, ninf, label="n_inf"); plt.plot(V, ninf ** 4, label="n_inf^4 (the conductance)")
plt.xlabel("V (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fitted curve minimises squared error on the grid, with a slope factor between 15 and 20 mV"
    err = lambda vh, k: np.sum((ninf - 1 / (1 + np.exp(-(V - vh) / k))) ** 2)
    assert abs(err(V_half, k_slope) - min(err(vh, k) for vh in np.arange(-80, 0.1, 0.5) for k in np.arange(5, 30.01, 0.1))) < 1e-9
    assert 15 < k_slope < 20

def test_charge():
    "z is V_T at 6.3 C divided by the slope factor, between 1 and 2 elementary charges"
    assert abs(z - 24.08 / k_slope) < 1e-9 and 1 < z < 2

test_fit()
test_charge()
print("All checks passed")

### Exercise: Fitting the power

The array `g` holds a noisy potassium conductance after a step from -65 to +20 mV. For each power p from 1 to 6, fit the time constant by grid search with the model gK (a - (a - b) e^(-t/τ))^p, where a and b are the p-th roots of the final and initial conductances divided by gK. Report the error for each p and the best p and τ.

In [ ]:
r = np.random.default_rng(30)
gK = 36.0
t = np.arange(0, 20, 0.01)
n0, ninf, tau_true = 0.3177, 0.9457, 1.2600
g = gK * (ninf - (ninf - n0) * np.exp(-t / tau_true)) ** 4 + r.normal(0, 0.3, t.size)
g_end, g0 = gK * ninf ** 4, gK * n0 ** 4              # final and initial values, known from the records

def model(p, tau):
    a, b = (g_end / gK) ** (1 / p), (g0 / gK) ** (1 / p)
    return ...

taus = np.linspace(0.3, 5, 471)
results = {}
for p in range(1, 7):
    errs = [np.sum((g - model(p, tau)) ** 2) for tau in taus]
    results[p] = (min(errs), taus[int(np.argmin(errs))])
    print(f"p = {p}: error {results[p][0]:8.1f}, tau {results[p][1]:.2f} ms")
p_best = ...
tau_best = results[p_best][1]
plt.plot(t, g, ".", ms=1, label="data")
for p in (1, 4):
    plt.plot(t, model(p, results[p][1]), label=f"p = {p}")
plt.xlim(0, 6); plt.xlabel("ms"); plt.ylabel("g_K (mS/cm^2)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_model():
    "model gives (a - (a - b) e^(-t/tau))^p with a and b the p-th roots of the final and initial conductances"
    a, b = (g_end / gK) ** 0.25, (g0 / gK) ** 0.25
    assert np.allclose(model(4, 1.3), gK * (a - (a - b) * np.exp(-t / 1.3)) ** 4)

def test_best():
    "the best power is 4 and its time constant is within 0.1 ms of the true 1.26 ms"
    assert p_best == 4 and abs(tau_best - 1.26) < 0.1

test_model()
test_best()
print("All checks passed")

### Exercise: Recovering a rate function

The arrays `ninf_fit` and `tau_fit` hold steady states and time constants fitted at 14 voltages, with 5% errors. Compute `alpha` and `beta`, then fit a straight line to ln(beta) against V + 65 and recover A and B in beta = A exp(-(V + 65) / B).

In [ ]:
r2 = np.random.default_rng(52)
Vs = np.arange(-80, 51, 10.0)
an, bn = rates_n(Vs)
ninf_fit = np.clip(an / (an + bn) * (1 + 0.05 * r2.normal(size=Vs.size)), 0, 0.99)
tau_fit = 1 / (an + bn) * (1 + 0.05 * r2.normal(size=Vs.size))

alpha = ...
beta = ...
slope, intercept = np.polyfit(Vs + 65, np.log(beta), 1)
A = ...
B = ...
print(f"beta_n = {A:.3f} exp(-(V + 65) / {B:.1f});  Hodgkin and Huxley: 0.125 exp(-(V + 65) / 80)")
plt.semilogy(Vs, beta, "o", label="beta from fits"); plt.semilogy(Vs, A * np.exp(-(Vs + 65) / B), label="fit")
plt.semilogy(Vs, alpha, "s", label="alpha from fits"); plt.xlabel("V (mV)"); plt.ylabel("rate (1/ms)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_rates():
    "alpha and beta are computed from n_inf and tau"
    assert np.allclose(alpha, ninf_fit / tau_fit) and np.allclose(beta, (1 - ninf_fit) / tau_fit)

def test_fit():
    "the exponential fit recovers A within 15% of 0.125 per ms and B within 15% of 80 mV"
    assert abs(A - 0.125) / 0.125 < 0.15 and abs(B - 80) / 80 < 0.15

test_rates()
test_fit()
print("All checks passed")